# Metamorphic Testing for Traditional Software (SuT: `calculate_shipping_fee`)

This exercise matches the module notes on **metamorphic testing**: a **property-based** way to mitigate the **oracle problem** by checking **necessary relational properties** among several input–output pairs, instead of insisting on one exact expected value for every test.

Concretely, you start from a **source test case** (input `x`, observe `f(x)`), derive **follow-up test cases** by transforming `x` into `x'`, observe `f(x')`, and verify a **metamorphic relation (MR)** that must hold between `(x, f(x))` and `(x', f(x'))`. Those relations act as a **pseudo oracle** when a full oracle is costly or impractical—the same role described in the lecture.

We use the same shipping-fee SuT as in prior modules so you can connect MT to requirements, search-based generation, and oracle trade-offs you have already seen.

In [4]:
def calculate_shipping_fee(
    order_total: int,
    weight_kg: float,
    distance_km: int,
    is_island: bool,
    membership: str,
    coupon_type: str,
) -> int:
    """SuT: shipping fee from business rules (integer fee, clamped at zero)."""
    fee = 0

    if order_total < 40000:
        fee += 3000

    if weight_kg <= 5:
        fee += 0
    elif weight_kg <= 20:
        fee += 2000
    else:
        fee += 5000

    if distance_km <= 10:
        fee += 0
    elif distance_km <= 50:
        fee += 1000
    else:
        fee += 3000

    if is_island:
        fee += 4000

    if membership == "WOW":
        fee = fee // 2

    if coupon_type == "NEW_USER":
        fee -= 2000

    return max(fee, 0)


# Smoke check
print("smoke:", calculate_shipping_fee(30000, 10.0, 20, False, "WOW", "NONE"))

smoke: 3000


## Define metamorphic relations for `calculate_shipping_fee`

The lecture stresses that MRs are often **tightly coupled to domain knowledge**. Here, the domain is the shipping-fee policy. The business rules below are the same specification narrative used in the oracle-problem exercise; MRs in the next table are **logical consequences** of these rules (plus the given computation order: surcharges accumulate, then `WOW` halves the subtotal, then `NEW_USER` subtracts, then clamp). **Selecting** non-trivial, requirement-grounded MRs—rather than only trivial identities—is what makes MT effective in practice.

- If `order_total` is **less than** 40,000, charge a base shipping fee of 3,000.
- If `weight_kg` is **5 or less**, apply no weight surcharge.
- If `weight_kg` is **over 5 and up to 20**, apply a weight surcharge of 2,000.
- If `weight_kg` is **over 20**, apply a weight surcharge of 5,000.
- If `distance_km` is **10 or less**, apply no distance surcharge.
- If `distance_km` is **over 10 and up to 50**, apply a distance surcharge of 1,000.
- If `distance_km` is **over 50**, apply a distance surcharge of 3,000.
- If `is_island` is true, apply an island surcharge of 4,000.
- If membership is `WOW`, halve the fee computed so far.
- If coupon type is `NEW_USER`, subtract 2,000 from the fee.
- The final fee must not be negative; clamp values below 0 to 0.

**Metamorphic relations used in this notebook**

| ID | Name | Idea (from requirements) | Source → follow-up | Expected relation |
| --- | --- | --- | --- | --- |
| MR-1 | Non-negative fee | Final fee is clamped at 0. | *identity* (any seed `x`) | $f(x) \geq 0$ |
| MR-2 | Island surcharge monotonicity | Island only adds a non-negative surcharge before halving. | Same `x`, but follow-up sets `is_island=True` when the source had `False` | $f(x') \geq f(x)$ |
| MR-3 | `WOW` is not worse | Halving applies to the entire pre-coupon subtotal and never increases it. | Same fields except `membership`: compare `"NONE"` (source) vs `"WOW"` (follow-up) | $f(\text{NONE}) \geq f(\text{WOW})$ |
| MR-4 | `NEW_USER` is not worse | Coupon only subtracts a fixed amount before clamp. | Same fields except `coupon_type`: compare `"NONE"` vs `"NEW_USER"` | $ f(\text{NONE}) \geq f(\text{NEW\_USER})$ |
| MR-5 | Drop base fee across 40k | Crossing from below 40k to at or above removes exactly the 3,000 base component (with `NONE` / `NONE`). | Only `order_total` changes: $30,000 \to 50,000$ (example pair) | $f(x) - f(x') = 3,000$ |
| MR-6 | Weight middle tier | Moving from the “≤5 kg” band to the “(5, 20] kg” band adds +2,000 before discounts, if other surcharges are fixed. | Same except `weight_kg`: $4.0 \to 10.0$ with `order_total \geq 40\,000`, short distance | $f(x') - f(x) = 2,000$ |
| MR-7 | Distance middle tier | Moving from “≤10 km” to “(10, 50] km” adds +1,000 before discounts, if other surcharges are fixed. | Same except `distance_km`: $8 \to 15$ with heavy order to avoid base-fee interaction | $f(x') - f(x) = 1\,000$ |
| MR-8 |  |  |  |  |
| MR-9 |  |  |  |  |
| MR-10 |  |  |  |  |

*(Define **MR-8–MR-10** above with your own metamorphic relations derived from the business rules.)*

Some MRs are **inequalities** (robust when the final clamp rarely binds). Others are **equalities on deltas**; those are stated under **restrictions** (`NONE` / `NONE`, chosen numeric bands) so the expected difference is not entangled with the `WOW` half or coupon subtraction.


## Implementing MR checks and source test cases

This chapter instantiates steps: we **generate sources**, treat `calculate_shipping_fee(...)` as the **observed** output for each input dict, **derive follow-ups** inside each MR helper (often as a copy of the source with one field changed), then **compare** `f(x)` and `f(x')`.

We encode each MR as a small Python function that (1) builds `x` and `x'`, (2) calls the SuT twice to obtain observed fees, and (3) asserts the relation.

**Source test cases** in the Chapter 4 script are plain dictionaries declared **one at a time** before each MR that needs a seed; you can also draw ideas from random or search-based generation in other modules. The metamorphic layer is reusable: the same MR helpers can be called with many different sources.

**MR-1–MR-4** below are complete reference checks. **MR-5–MR-10** are empty stubs in `mr5_todo()` … `mr10_todo()` (`pass`): implement them to match your table above, then run the sequential test script in the next chapter (it calls MR-1 … MR-10 in order).


In [6]:
from copy import deepcopy


def F(inp: dict) -> int:
    """Shorthand: call the SuT on a single input dict."""
    return calculate_shipping_fee(**inp)


def mr1_non_negative(seed: dict) -> None:
    assert F(seed) >= 0, "MR-1 failed: fee should never be negative after clamp"


def mr2_island_not_cheaper(seed: dict) -> None:
    if seed["is_island"]:
        return  # no strict mainland→island step to apply
    x = deepcopy(seed)
    x_prime = deepcopy(seed)
    x_prime["is_island"] = True
    assert F(x_prime) >= F(x), "MR-2 failed: island delivery should not reduce fee vs mainland"


def mr3_wow_not_worse(seed: dict) -> None:
    x = deepcopy(seed)
    x["membership"] = "NONE"
    x_prime = deepcopy(seed)
    x_prime["membership"] = "WOW"
    assert F(x) >= F(x_prime), "MR-3 failed: WOW membership should not increase fee vs NONE"


def mr4_new_user_not_worse(seed: dict) -> None:
    x = deepcopy(seed)
    x["coupon_type"] = "NONE"
    x_prime = deepcopy(seed)
    x_prime["coupon_type"] = "NEW_USER"
    assert F(x) >= F(x_prime), "MR-4 failed: NEW_USER coupon should not increase fee vs NONE"


def mr5_todo() -> None:
    """TODO: MR-5 — build source `x` and follow-up `x_prime`; assert the relation on `F` (see Chapter 2)."""
    pass


def mr6_todo() -> None:
    """TODO: MR-6 — same pattern (`x`, `x_prime`)."""
    pass


def mr7_todo() -> None:
    """TODO: MR-7 — same pattern (`x`, `x_prime`)."""
    pass


def mr8_todo() -> None:
    """TODO: MR-8 — define `x` and `x_prime` per your Chapter 2 table."""
    pass


def mr9_todo() -> None:
    """Student: MR-9 — define `x` and `x_prime` per your Chapter 2 table."""
    pass


def mr10_todo() -> None:
    """TODO: MR-10 — define `x` and `x_prime` per your Chapter 2 table."""
    pass


print("MR helper functions loaded (define each source dict in the Chapter 4 run cell).")

MR helper functions loaded (define each source dict in the Chapter 4 run cell).


## Chapter 4. Run metamorphic testing

This step follows the lecture workflow: execute the SuT on follow-ups, **compare** source and follow-up outputs against each MR, and **treat any failed check as a potential fault** in the implementation or in the MR specification.

The cell below is a **straight-line script** (not wrapped in a function): for **MR-1 … MR-4** it defines one **source** dict, then immediately runs that MR; **MR-5 … MR-10** follow in order (their sources live inside the helper bodies once you implement them, or you add dicts here). No lists, no loops—just ten explicit steps.

Watch how combining **several sources** with **several relations** gives richer behavioral evidence than a tiny list of manual expected fees.


In [7]:
# --- Sequential metamorphic checks (MR-1 … MR-10); each step defines its own source where needed ---

print("--- MR-1 ---")
source_mr1 = {
    "order_total": 12000,
    "weight_kg": 3.0,
    "distance_km": 5,
    "is_island": False,
    "membership": "NONE",
    "coupon_type": "NONE",
}
mr1_non_negative(source_mr1)
print("MR-1 OK")

print("\n--- MR-2 ---")
source_mr2 = {
    "order_total": 12000,
    "weight_kg": 3.0,
    "distance_km": 5,
    "is_island": False,
    "membership": "NONE",
    "coupon_type": "NONE",
}
mr2_island_not_cheaper(source_mr2)
print("MR-2 OK")

print("\n--- MR-3 ---")
source_mr3 = {
    "order_total": 39999,
    "weight_kg": 20.0,
    "distance_km": 50,
    "is_island": False,
    "membership": "WOW",
    "coupon_type": "NEW_USER",
}
mr3_wow_not_worse(source_mr3)
print("MR-3 OK")

print("\n--- MR-4 ---")
source_mr4 = {
    "order_total": 80000,
    "weight_kg": 1.0,
    "distance_km": 3,
    "is_island": False,
    "membership": "NONE",
    "coupon_type": "NEW_USER",
}
mr4_new_user_not_worse(source_mr4)
print("MR-4 OK")

print("\n--- MR-5 ---")
mr5_todo()
print("MR-5 OK")

print("\n--- MR-6 ---")
mr6_todo()
print("MR-6 OK")

print("\n--- MR-7 ---")
mr7_todo()
print("MR-7 OK")

print("\n--- MR-8 ---")
mr8_todo()
print("MR-8 OK")

print("\n--- MR-9 ---")
mr9_todo()
print("MR-9 OK")

print("\n--- MR-10 ---")
mr10_todo()
print("MR-10 OK")

print("\nDone: MR-1 … MR-10 executed in order.")


--- MR-1 ---
MR-1 OK

--- MR-2 ---
MR-2 OK

--- MR-3 ---
MR-3 OK

--- MR-4 ---
MR-4 OK

--- MR-5 ---
MR-5 OK

--- MR-6 ---
MR-6 OK

--- MR-7 ---
MR-7 OK

--- MR-8 ---
MR-8 OK

--- MR-9 ---
MR-9 OK

--- MR-10 ---
MR-10 OK

Done: MR-1 … MR-10 executed in order.


### Reflection

MT is **domain-specific**. What kinds of MR can be defined in different applications?

- **Numerical / scientific**
  - *Example:* For `sin`, **odd symmetry** should hold: `sin(-x)` matches `-sin(x)` up to a tiny tolerance, which catches sign or quadrant bugs without a hand-computed reference for every `x`.
  - *Else?* 
- **Search engines**
  - *Example:* If you **narrow** the query by adding another **AND** conjunct (extra required keyword/constraint), the **number of hits** should be **at most** that of the looser query: $|R_{q \land q'}| \le |R_q|$ (result set can only **shrink or stay the same**).
  - *Else?* 
- **Robotics / mobility**
  - *Example:* The map is **symmetric** like a hallway mirrored left–right, and the robot cares only about distance, not “which side.” Then a **sensible route** on the left half should look like the **mirror** of a sensible route on the flipped map (same idea as folding paper in half).
  - *Else?* 
- **Compilers / optimizers**
  - *Example:* **Statement reorder:** swapping the order of **dependency-free** statements must **not** change **observable output**; if it does, suspect wrong **dependency analysis** or unsafe **optimization reordering**.
  - *Else?* 
- **Enterprise rules** (billing, tax, HR policy engines)
  - *Example:* **Duration-based billing:** with a fixed published rate card (e.g. parking, compute usage, support hours), charging for a **longer** time window in the **same** tier must not produce a **lower** total than a **shorter** window—violations hint at **rounding**, **boundary**, or **proration** bugs.
  - *Else?* 

> **Takeaway:** MRs ≈ **pseudo oracle** when exact outputs are hard; pick **meaningful**, **varied** checks.
